In [ ]:
# Install dependencies if needed
# %pip install -r ../requirements.txt
# dbutils.library.restartPython()

In [ ]:
import sys
import time
from pprint import pprint

sys.path.insert(0, '..')

from multiAgentSystem.graph import build_graph
from multiAgentSystem.state import AgentState
from experiments.mlflow_setup import (
    setup_experiment,
    create_agent_run,
    log_agent_metrics,
    log_state_snapshot,
    enable_autologging
)
from experiments.mock_data import WORKFLOW_TEST_SCENARIOS, get_workflow_scenario

print("✓ Imports successful")

In [ ]:
# Setup MLflow experiment
enable_autologging()
experiment_id = setup_experiment("full_workflow")
print(f"Experiment ID: {experiment_id}")

In [ ]:
# Configuration: Update this path to your actual Spark logs
REAL_LOGS_PATH = "/Volumes/amruthcatalogtest/default/testsparklogs/sample/"

print(f"Using logs path: {REAL_LOGS_PATH}")
print("\n⚠️ If tests fail, ensure the logs path exists and contains Spark log files.")

In [ ]:
# Create the graph once
graph = build_graph()
print("✓ Graph created successfully")

In [ ]:
def run_full_workflow_test(
    scenario_name: str, 
    logs_path_override: str = None,
    verbose: bool = True,
    generate_pdf: bool = False
):
    """
    Run a complete workflow test with MLflow tracking.
    
    Args:
        scenario_name: Name of test scenario from WORKFLOW_TEST_SCENARIOS
        logs_path_override: Override the logs path from scenario
        verbose: Print detailed progress
        generate_pdf: Whether to generate PDF report at end
    """
    scenario = get_workflow_scenario(scenario_name)
    initial_state: AgentState = scenario["initial_state"].copy()
    expected = scenario["expected"]
    
    # Override logs path if provided
    if logs_path_override:
        initial_state["logs_path"] = logs_path_override
    
    with create_agent_run("full_workflow", scenario=scenario_name) as run:
        log_state_snapshot(initial_state, prefix="input")
        
        import mlflow
        mlflow.log_param("scenario_type", scenario["issue_type"])
        mlflow.log_param("generate_pdf", generate_pdf)
        mlflow.log_param("logs_path", initial_state.get("logs_path", "N/A"))
        
        if verbose:
            print(f"\n{'='*60}")
            print(f"Running: {scenario_name}")
            print(f"Description: {scenario['description']}")
            print(f"{'='*60}")
        
        start_time = time.time()
        agent_transitions = []
        
        try:
            # Run the graph with streaming to track transitions
            final_state = None
            for step_output in graph.stream(initial_state, stream_mode="values"):
                final_state = step_output
                current_agent = step_output.get("next_agent", "unknown")
                agent_transitions.append(current_agent)
                
                if verbose:
                    print(f"  -> {current_agent}")
            
            success = True
        except Exception as e:
            final_state = {"error": str(e)}
            success = False
            if verbose:
                print(f"  ❌ Error: {e}")
        
        latency_ms = (time.time() - start_time) * 1000
        
        # Extract metrics from final state
        evidence_map = final_state.get("evidence_map", {}) if final_state else {}
        confidence = final_state.get("confidence", 0.0) if final_state else 0.0
        iteration_count = final_state.get("iteration", 0) if final_state else 0
        draft = final_state.get("draft", {}) if final_state else {}
        final_report = str(draft.get("rca", "")) if draft else ""
        
        log_agent_metrics(
            latency_ms=latency_ms,
            success=success,
            additional_metrics={
                "total_agent_transitions": len(agent_transitions),
                "iteration_count": iteration_count,
                "final_confidence": confidence,
                "evidence_patterns": len(evidence_map),
                "final_report_length": len(final_report),
                "reached_final": 1.0 if "final" in agent_transitions else 0.0,
            }
        )
        
        # Log transition history
        mlflow.log_param("agent_transitions", " -> ".join(agent_transitions[:20]))  # Limit for param size
        
        log_state_snapshot(final_state or {}, prefix="output")
        
        # Evaluate against expected outcomes
        passed = True
        
        if expected.get("min_confidence"):
            if confidence < expected["min_confidence"]:
                passed = False
                
        if expected.get("has_evidence") and not evidence_map:
            passed = False
            
        if expected.get("has_final_report") and not final_report:
            passed = False
        
        mlflow.log_metric("test_passed", 1.0 if passed else 0.0)
        
        # Generate PDF if requested
        pdf_success = False
        if generate_pdf and draft:
            try:
                from multiAgentSystem.tools.pdf_report_tool import generate_pdf_report
                pdf_path = generate_pdf_report(
                    final_state,
                    output_path=f"/tmp/rca_report_{scenario_name}.pdf"
                )
                if pdf_path:
                    mlflow.log_artifact(pdf_path)
                    pdf_success = True
            except Exception as e:
                print(f"  ⚠️ PDF generation failed: {e}")
        
        mlflow.log_metric("pdf_generated", 1.0 if pdf_success else 0.0)
        
        if verbose:
            status = "✅ PASSED" if passed else "❌ FAILED"
            print(f"\n{status} - {scenario_name}")
            print(f"  Total latency: {latency_ms/1000:.2f}s")
            print(f"  Agent transitions: {len(agent_transitions)}")
            print(f"  Iterations: {iteration_count}")
            print(f"  Confidence: {confidence:.2f}")
            print(f"  Evidence patterns: {len(evidence_map)}")
            print(f"  Report length: {len(final_report)} chars")
            if generate_pdf:
                print(f"  PDF generated: {pdf_success}")
        
        return final_state, passed, latency_ms, agent_transitions

## Test 1: OOM Investigation

Full workflow investigating OutOfMemoryError symptoms.

In [ ]:
result_1, passed_1, latency_1, transitions_1 = run_full_workflow_test(
    "oom_investigation",
    logs_path_override=REAL_LOGS_PATH,
    generate_pdf=True
)

## Test 2: Executor Loss Investigation

Full workflow investigating executor lost events and network issues.

In [ ]:
result_2, passed_2, latency_2, transitions_2 = run_full_workflow_test(
    "executor_loss_investigation",
    logs_path_override=REAL_LOGS_PATH,
    generate_pdf=True
)

## Test 3: GC Pressure Investigation

Full workflow investigating garbage collection overhead issues.

In [ ]:
result_3, passed_3, latency_3, transitions_3 = run_full_workflow_test(
    "gc_pressure_investigation",
    logs_path_override=REAL_LOGS_PATH,
    generate_pdf=True
)

## Summary

In [ ]:
print("=" * 70)
print("FULL WORKFLOW TEST SUMMARY")
print("=" * 70)

tests = [
    ("oom_investigation", passed_1, latency_1, len(transitions_1)),
    ("executor_loss_investigation", passed_2, latency_2, len(transitions_2)),
    ("gc_pressure_investigation", passed_3, latency_3, len(transitions_3)),
]

total_passed = sum(1 for _, passed, _, _ in tests if passed)
avg_latency = sum(lat for _, _, lat, _ in tests) / len(tests)
avg_transitions = sum(trans for _, _, _, trans in tests) / len(tests)

for name, passed, latency, transitions in tests:
    status = "✅" if passed else "❌"
    print(f"  {status} {name}:")
    print(f"      Latency: {latency/1000:.2f}s | Transitions: {transitions}")

print("=" * 70)
print(f"Total: {total_passed}/{len(tests)} passed")
print(f"Average latency: {avg_latency/1000:.2f}s")
print(f"Average transitions: {avg_transitions:.1f}")
print("=" * 70)

## Visualize Agent Transitions

In [ ]:
def visualize_transitions(transitions: list, title: str):
    """Simple text visualization of agent transitions."""
    print(f"\n{title}")
    print("-" * 40)
    for i, agent in enumerate(transitions):
        prefix = "└─>" if i == len(transitions) - 1 else "├─>"
        print(f"  {prefix} {agent}")

visualize_transitions(transitions_1, "OOM Investigation Flow")
visualize_transitions(transitions_2, "Executor Loss Investigation Flow")
visualize_transitions(transitions_3, "GC Pressure Investigation Flow")

## View MLflow Experiments

Navigate to the MLflow UI in Databricks to see:
- Run comparisons across scenarios
- Latency trends
- Confidence score distributions
- Generated PDF artifacts

In [ ]:
import mlflow

# Get experiment URL (Databricks)
experiment = mlflow.get_experiment_by_name("/Shared/spark-rca/full-workflow")
if experiment:
    print(f"Experiment ID: {experiment.experiment_id}")
    print(f"\n📊 View in MLflow UI to compare runs and download artifacts.")